# VoiceCraft Studio & Telegram Bot (Free Google Colab T4 GPU)
Run your Hinglish Voice Cloning Studio and Telegram Bot with **Free NVIDIA T4 GPU acceleration**.

- **Speed**: ~1 second per sentence (10x faster than CPU)
- **Cost**: 100% Free via Google Colab + Cloudflare Tunnel
- **Features**: Web Studio, Telegram Bot, Multilingual TTS (v3), Hinglish language routing

---
### Quick Instructions:
1. In Google Colab top menu, click **Runtime** > **Change runtime type** > Select **T4 GPU** > Save.
2. Run **Step 1** to verify GPU.
3. Run **Step 2** to install dependencies.
4. Run **Step 3** to paste your Telegram Bot Token.
5. Run **Step 4** to launch and get your public web link!

In [ ]:
# @title Step 1: Check GPU Acceleration
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU is not enabled! Go to Runtime -> Change runtime type -> T4 GPU.')


In [ ]:
# @title Step 2: Clone Repository & Install Dependencies
import os
import shutil
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'

# Ensure clean base directory
%cd /content
if os.path.exists('/content/VoiceCraft/VoiceCraft'):
    shutil.rmtree('/content/VoiceCraft/VoiceCraft', ignore_errors=True)

if not os.path.exists('/content/VoiceCraft'):
    !git clone https://github.com/ajeet3301/VoiceCraft.git
else:
    !git -C /content/VoiceCraft pull
%cd /content/VoiceCraft

# Install Linux system packages
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 > /dev/null

# Fix NumPy for Numba compatibility (<2.4.0) and TensorFlow collision
!pip install -q "numpy<2.4.0"
!pip install -q -U "ml_dtypes>=0.5.0"
!pip install -q -r requirements_colab.txt
!pip install -q -e . --no-deps

# Download portable Linux Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print('\nSetup completed successfully!')


In [ ]:
# @title Step 3: Configure Telegram Bot & Model Settings
# @markdown Enter your Telegram bot credentials below (optional):
%cd /content/VoiceCraft
TELEGRAM_BOT_TOKEN = 'your_telegram_bot_token_here'  # @param {type:'string'}
MODEL_VERSION = 'v3'  # @param ['v3', 'v2', 'v1']
ADMIN_USER_ID = ''  # @param {type:'string'}

env_lines = [
    f'BOT_TOKEN={TELEGRAM_BOT_TOKEN.strip()}',
    f'TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN.strip()}',
    f'CHATTERBOX_MULTILINGUAL_T3_MODEL={MODEL_VERSION}',
]
if ADMIN_USER_ID.strip():
    env_lines.append(f'ADMIN_USER_ID={ADMIN_USER_ID.strip()}')

with open('.env', 'w', encoding='utf-8') as f:
    f.write('\n'.join(env_lines) + '\n')

print(f'Configuration saved. Model set to: {MODEL_VERSION}')
if TELEGRAM_BOT_TOKEN.strip() and TELEGRAM_BOT_TOKEN.strip() != 'your_telegram_bot_token_here':
    print('Telegram Bot is enabled and will start automatically with the server.')
else:
    print('No Telegram Bot token provided. Only Web Studio UI will be active.')


In [ ]:
# @title Step 4: Launch VoiceCraft Web Studio & Telegram Bot
%cd /content/VoiceCraft
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
import subprocess
import time
import re
from IPython.display import display, HTML

# Start Cloudflare Tunnel in background
print('Creating Cloudflare Tunnel for public access...')
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('=' * 60)
    print(f'  YOUR APP IS LIVE AT: {public_url}')
    print('=' * 60)
    display(HTML(f'''
    <div style="background-color: #FAF9F5; border: 1px solid #C96442; border-radius: 8px; padding: 20px; margin: 15px 0; font-family: sans-serif;">
        <h3 style="color: #1F1E1D; margin-top: 0; font-family: Georgia, serif;">VoiceCraft Studio &amp; Bot Online</h3>
        <p style="color: #555; margin-bottom: 15px;">Open this public URL on your phone or PC browser:</p>
        <a href="{public_url}" target="_blank" style="background-color: #C96442; color: #FAF9F5; text-decoration: none; padding: 10px 22px; border-radius: 6px; font-weight: 600; display: inline-block;">
            Open VoiceCraft Studio &#8599;
        </a>
    </div>
    '''))
else:
    print('Could not automatically detect public link. Check tunnel output.')

# Launch server.py (auto-starts Telegram bot if token is configured)
print('\nStarting VoiceCraft server (FastAPI on port 7860)...')
!python server.py
